# STEP 11 — 오답이 **어디로** 가는지 (학습 없음, ~20분)

STEP 10 에서 정상 사진의 33.3% 가 병원으로 갔습니다. 클래스별 precision·recall
만으로는 그 1,233장이 **어느 병변으로** 갔는지 알 수 없습니다.

이 노트북은 **학습을 하지 않습니다.** 저장해둔 가중치로 추론만 다시 돌려
혼동행렬의 실제 칸을 보고, 그 사진들을 눈으로 봅니다 (멘토 피드백 3·5번).

| Kaggle 입력 | |
|---|---|
| `dogskin-f320` | 1단계 |
| `dogskin-m25` | 2단계 |
| **06 의 release** | 가중치 — 이건 **꼭 붙여야 합니다** |

## 🚨 판단은 val 로 합니다

여기서 나온 걸 보고 설정을 고치면 **holdout 이 오염됩니다.** 그래서:

| | 쓰는 곳 |
|---|---|
| **val** | 여기서 보는 숫자·사진. 다음 실험을 정하는 근거 |
| holdout | 같은 모양인지 **확인만**. 여기 보고 설정을 고르지 않습니다 |

## 무엇이 나오나

1. 자주 헷갈리는 칸 — 정답 → 예측, 장수 순
2. 헛알림의 행선지 — 정상이 어느 병변으로 갔나 (**주변합 역산이 아닌 실제 칸**)
3. 놓친 병변 — "괜찮아요" 라고 안심시킨 것이 어느 병변이었나 ← 가장 위험
4. 그 사진들을 한 판에 깔아 눈으로 보기


In [ ]:
# ── 0. 환경 준비 (Colab / Kaggle 공통) ──────────────────────────
# 이 셀 하나가 리포 동기화 → 패키지 설치 → 환경 감지까지 다 합니다.
# 리포를 직접 다운로드하거나 드라이브에 올릴 필요 없습니다.
# 다시 실행하면 항상 최신 코드로 맞춰집니다 (로컬 수정은 덮어씁니다).
import os, sys, subprocess

REPO   = "https://github.com/gayeoniee/deeplearning_test.git"
BRANCH = "main"
NAME   = "deeplearning_test"
_cwd   = os.getcwd()
if os.path.basename(_cwd) == NAME and os.path.isdir(os.path.join(_cwd, ".git")):
    DIR = _cwd            # 이미 리포 안에서 재실행 중 (중첩 clone 방지)
else:
    # ⚠️ Kaggle 을 먼저 봅니다. Kaggle 이미지에도 /content 가 있어서
    #    /content 를 먼저 보면 Kaggle 세션인데 /content 에 clone 합니다.
    BASE = ("/kaggle/working" if os.path.isdir("/kaggle/working")
            else "/content" if os.path.isdir("/content") else _cwd)
    DIR = os.path.join(BASE, NAME)

if os.path.isdir(os.path.join(DIR, ".git")):
    # 이미 받아둔 경우: 최신으로 강제 동기화 (shallow clone 에서도 안전)
    subprocess.run(["git", "-C", DIR, "fetch", "--depth", "1", "origin", BRANCH], check=False)
    subprocess.run(["git", "-C", DIR, "reset", "--hard", f"origin/{BRANCH}"], check=False)
else:
    subprocess.run(["git", "clone", "-b", BRANCH, "--depth", "1", REPO, DIR], check=True)

os.chdir(DIR)
if DIR not in sys.path:
    sys.path.insert(0, DIR)

# ⚠️ 중요: 이미 import 된 src.* 는 파이썬이 캐시하고 있어서
#    파일을 갱신해도 옛날 코드가 그대로 쓰입니다. 캐시를 비웁니다.
for _m in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[_m]

print("작업 디렉터리:", os.getcwd())
print("코드 버전   :", subprocess.run(["git", "-C", DIR, "log", "--oneline", "-1"],
                                      capture_output=True, text=True).stdout.strip())

# 패키지 설치는 **uv 로 통일**합니다 (pip 보다 훨씬 빠릅니다).
# ⚠️ Colab/Kaggle 이미지에는 uv 가 없어서, uv 자체만 pip 로 한 번 받습니다.
#    --system = 가상환경을 새로 만들지 않고 이미 있는 파이썬에 그대로 설치.
#    (torch/numpy/pandas 는 이미 깔려 있으므로 여기서 안 건드립니다)
# albumentations 는 import 할 때마다 PyPI 에 버전 확인 요청을 보냅니다.
# Kaggle 은 외부 네트워크가 막혀 있어 타임아웃(2초)만 기다리다 끝납니다 — 꺼둡니다.
os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"

_PKGS = ["timm", "imagehash", "pyarrow", "grad-cam", "albumentations"]
_ok = False
if subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"],
                  check=False).returncode == 0:
    _ok = subprocess.run([sys.executable, "-m", "uv", "pip", "install", "-q",
                          "--system", *_PKGS], check=False).returncode == 0
if not _ok:
    print("[env] uv 로 설치하지 못해 pip 으로 대체합니다")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_PKGS], check=False)

# 한글 그래프 폰트 (Colab 기본에는 한글이 없어 □ 로 나옵니다)
_font = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(_font):
    subprocess.run(["apt-get", "install", "-y", "-qq", "fonts-nanum"], check=False)
try:
    import matplotlib.pyplot as plt, matplotlib.font_manager as fm
    fm.fontManager.addfont(_font)
    plt.rcParams["font.family"] = "NanumGothic"
    plt.rcParams["axes.unicode_minus"] = False
except Exception:
    pass

MY_NOTEBOOK_VERSION = "2026-08-24.1"   # ★ 이 셀(=이 .ipynb)의 버전

from src import env
from src.config import CFG, CLASSES, CLASS_KO
E = env.describe()
env.set_seed(42)

# 환경 판정이 이상하면(예: Kaggle 인데 colab 이라고 나오면) 근거를 봅니다
if E.env != "local":
    env.diagnose()

# ⚠️ 노트북 셀은 git pull 로 갱신되지 않습니다 (src/ 만 최신이 됩니다).
#    낡은 .ipynb 를 몇 시간 돌리고 나서 알게 되면 늦으므로 지금 확인합니다.
from src.config import NOTEBOOK_VERSION as _repo_nb
if MY_NOTEBOOK_VERSION != _repo_nb:
    print("\n" + "!" * 62)
    print(f"⚠️ 이 노트북이 낡았습니다 — 내 셀 {MY_NOTEBOOK_VERSION} / 리포 {_repo_nb}")
    print("   src/ 는 최신이지만 **셀 내용은 예전 것**입니다.")
    print("   GitHub 에서 notebooks/*.ipynb 를 다시 받아 Import 하세요:")
    print("   Kaggle → File → Import Notebook / Colab → 파일 → 노트 업로드")
    print("!" * 62 + "\n")
else:
    print(f"[nb] 노트북 최신 ({_repo_nb})")


## 1. 데이터와 가중치 붙이기

⚠️ **release 를 안 붙이면 여기서 멈춥니다.** 이 노트북은 학습을 안 하므로
가중치를 만들어낼 방법이 없습니다. 06 실행의 Output 을 Private 데이터셋으로
만들어 Add Input 하세요.

In [ ]:
import json

import numpy as np
import pandas as pd
import torch

from src import (labels, split, crop, data, models, train, evaluate,
                 stages, errors, explain)
from src.config import (CFG, CLASSES, CLASS_KO, CLASSES_STAGE1,
                        NORMAL_LABEL, MODEL_BY_KEY)

env.load_prepared()

# ★ 이 노트북은 학습을 안 합니다 — 가중치를 어디선가 받아와야 합니다.
#   두 경로가 다 정상입니다:
#     · 06 을 돌린 **같은 세션**에서 이어서 열었다 → 작업 폴더에 이미 있음
#     · 새 세션에서 release 를 Add Input 했다        → 여기서 가져옴
#   그래서 "입력에 붙었나" 가 아니라 **가중치가 실제로 있나**로 판정합니다.
#   (앞엣것으로 막았더니 06 직후 같은 세션에서 열면 거부당했습니다)
_prev = train.import_previous_run(verbose=True)
_ck_dir = env.work_root() / "checkpoints"
_have = sorted(q.name for q in _ck_dir.glob("*") if (q / "best.pt").exists()) \
    if _ck_dir.exists() else []
if not _have:
    raise SystemExit(
        "❌ 쓸 수 있는 가중치가 없습니다 (이 노트북은 학습을 하지 않습니다).\n"
        "   · 06 을 돌린 같은 세션이면 그냥 이어서 돌리면 됩니다\n"
        "   · 새 세션이면 06 의 Output → New Dataset (Private) → Add Input\n"
        f"   찾아본 곳: {_ck_dir}")
print(f"\n[가중치] {_have}")

# ★ 학습은 안 하지만 **추론은 합니다** — val 7,751 + holdout 7,134 을 두 모델로.
#   env.py 실측: 검증 7,751장에 GPU 1~2분 vs CPU 약 40분. GPU 없이 시작하면
#   몇 시간 뒤에 알게 되므로 여기서 멈춥니다.
env.require_gpu()
DEV = "cuda" if torch.cuda.is_available() else "cpu"
W = env.work_root()
print("전체(val 7,751 + holdout 7,134)로 돕니다 — 약 20분")

df = labels.load(W / "manifests" / "manifest_final.parquet")
split.verify(df, fold=0, strict=True)
print(f"{len(df):,}행 / 개체 {df['animal_id'].nunique():,}마리")
print("붙어 있는 크롭 태그:", crop.available_tags())

## 2. 06 이 남긴 설정을 그대로 읽습니다

임계값·크롭·백본을 여기서 다시 고르지 않습니다. **06 이 정한 값을 그대로**
씁니다 — 다르게 쓰면 STEP 10 숫자와 비교가 안 됩니다.

In [ ]:
thr_json = W / "stage1_threshold.json"
if not thr_json.exists():
    raise SystemExit(f"❌ {thr_json} 이 없습니다. release 에 JSON 이 빠졌습니다.")
S = json.loads(thr_json.read_text(encoding="utf-8"))

THR1        = S["threshold"]
STAGE1_CROP = S["stage1_crop"]
BEST_CROP   = S["stage2_crop"]
IMG_SIZE    = S["img_size"]
EXP1, EXP2  = S["stage1_exp"], S["stage2_exp"]

# ⚠️ 백본은 **폴더 이름에서** 읽습니다. 하드코딩하면 1단계 effnetv2_s /
#    2단계 resnet50 처럼 서로 다를 때 shape mismatch 로 죽습니다 (실제로 당함).
KEY1 = train.model_key_from_exp(EXP1)
KEY2 = train.model_key_from_exp(EXP2)

print(f"1단계  {KEY1:<12} 크롭 {STAGE1_CROP:<6} exp {EXP1}")
print(f"2단계  {KEY2:<12} 크롭 {BEST_CROP:<6} exp {EXP2}")
print(f"입력 {IMG_SIZE}px / 임계값 {THR1:.4f}  (06 이 검증셋에서 정한 값)")

# 06 과 같은 방식 — cfg 는 2단계 백본 기준. 전처리는 eval_loader 에 넘긴
# model 로 결정되므로 두 단계가 각자 자기 전처리를 씁니다.
cfg = CFG(model_name=MODEL_BY_KEY[KEY2].timm_name, img_size=IMG_SIZE)
d = crop.switch_tag(df, BEST_CROP, verbose=False)
s1_all = stages.to_stage1(crop.switch_tag(df, STAGE1_CROP, verbose=False))
s2_all = stages.to_stage2(d)
split.verify(s1_all, fold=0, strict=True)
split.verify(s2_all, fold=0, strict=True)

In [ ]:
def _load(exp: str, key: str, n_cls: int):
    """체크포인트 하나를 그 이름이 말하는 백본으로 되살립니다."""
    train.restore_from_persist(exp, verbose=False)          # 영속 저장소 → 로컬
    ck = W / "checkpoints" / exp / "best.pt"
    if not ck.exists():
        have = sorted(q.name for q in (W / "checkpoints").glob("*")) \
            if (W / "checkpoints").exists() else []
        raise FileNotFoundError(f"{ck} 가 없습니다.\n  붙어 있는 것: {have}")
    if key not in MODEL_BY_KEY:
        raise SystemExit(f"❌ 모르는 백본 '{key}' (체크포인트 이름: {exp})\n"
                         f"   MODEL_ZOO 에 있는 키: {sorted(MODEL_BY_KEY)}")
    # ⚠️ models.load_checkpoint 는 백본이 다르면 shape mismatch 를 **설명과 함께**
    #    올려줍니다. 하드코딩으로 resnet50 을 넣던 시절에 실제로 죽었습니다.
    return models.load_checkpoint(str(ck), MODEL_BY_KEY[key], n_cls,
                                  device=DEV).to(DEV).eval()

m1 = _load(EXP1, KEY1, len(CLASSES_STAGE1))
m2 = _load(EXP2, KEY2, len(CLASSES))
print(f"두 모델 로드 완료 — 1단계 {KEY1} / 2단계 {KEY2}")

## 3. 같은 사진을 두 모델에 넣습니다

⚠️ 단계마다 크롭이 다릅니다. **각 모델에는 그 모델이 학습한 크롭**을 먹여야
합니다. `switch_tag` 는 경로만 다시 계산하므로 행 순서가 보존됩니다 —
그래도 아래에서 확인합니다. 순서가 어긋나면 점수가 **조용히** 엉망이 됩니다.

In [ ]:
def run_pipeline(view_s1, tag: str):
    """정상+병변 전체에 두 모델을 이어붙여 돌리고 pipeline_report 를 냅니다."""
    v2 = crop.switch_tag(view_s1, BEST_CROP, verbose=False) \
        if STAGE1_CROP != BEST_CROP else view_s1
    dl1, ds1 = data.eval_loader(view_s1, cfg, model=m1, classes=CLASSES_STAGE1)
    dl2, ds2 = data.eval_loader(v2, cfg, model=m2, classes=CLASSES)
    assert len(ds1.df) == len(ds2.df), f"{len(ds1.df)} vs {len(ds2.df)}"
    assert (ds1.df["image_name"].to_numpy() == ds2.df["image_name"].to_numpy()).all(), \
        "두 로더의 행 순서가 다릅니다"

    _, lg1, _ = train.evaluate_loader(m1, dl1, None, DEV, len(CLASSES_STAGE1),
                                      tta_hflip=True)
    _, lg2, _ = train.evaluate_loader(m2, dl2, None, DEV, len(CLASSES), tta_hflip=True)

    s1 = stages.stage1_scores(lg1)
    y  = ds1.df["label_orig"].to_numpy()
    rep = stages.pipeline_report(s1, lg2, y, threshold=THR1, show=False)
    pred, conf = stages.pipeline_predict(s1, lg2, THR1)

    out = ds1.df.copy()
    out["pred"] = [stages.PIPELINE_CLASSES[i] for i in pred]
    out["conf"] = conf
    out["s1_score"] = s1
    print(f"[{tag}] {len(out):,}장 "
          f"(정상 {(y == NORMAL_LABEL).sum():,} / 병변 {(y != NORMAL_LABEL).sum():,})  "
          f"최종 macro-F1 {rep['final_macro_f1']:.4f}")
    return rep, out

va_all = split.get_fold(s1_all, 0)[1]
rep_va, pv = run_pipeline(va_all, "val")            # ← 판단은 이걸로 합니다

# holdout 은 **같은 모양인지 확인만** 합니다 (여기 보고 설정을 고르지 않습니다)
ho_all = split.get_holdout(s1_all)   # 컬럼명은 is_holdout — 직접 만지지 않습니다
rep_ho, ph = run_pipeline(ho_all, "holdout")

## 4. 어느 칸으로 틀리나 — **실제 숫자**

지금까지는 클래스별 precision·recall 에서 **역산**했습니다. 그건 순 흐름만
보여주지 칸은 못 봅니다. 여기서부터가 진짜 값입니다.

In [ ]:
pairs_va = errors.confusion_pairs(rep_va, top=15)
print("\n※ 아래는 holdout — 같은 모양인지 **확인만** 합니다 (보고 고르지 않습니다)")
pairs_ho = errors.confusion_pairs(rep_ho, top=8)

In [ ]:
fa_va = errors.false_alarm_targets(rep_va)
fa_ho = errors.false_alarm_targets(rep_ho, show=False)
print(f"\n(holdout 확인) 헛알림 {fa_ho['false_alarms']:,}장 → "
      + " / ".join(f"{c} {n:,}" for c, n in
                   sorted(fa_ho["by_class"].items(), key=lambda kv: -kv[1]) if n))

# 역산으로 봤던 "A1·A2 가 89%" 가 실제 칸에서도 맞는지
_tot = fa_va["false_alarms"] or 1
_top2 = sum(sorted(fa_va["by_class"].values(), reverse=True)[:2])
print(f"\n★ 상위 2개 클래스가 헛알림의 {_top2 / _tot:.1%} 를 가져갑니다 (val)")

In [ ]:
ms_va = errors.miss_sources(rep_va)
ms_ho = errors.miss_sources(rep_ho, show=False)
print(f"\n(holdout 확인) 놓친 병변 {ms_ho['total']:,}장 → "
      + " / ".join(f"{c} {n:,}" for c, n in
                   sorted(ms_ho["by_class"].items(), key=lambda kv: -kv[1]) if n))

## 5. 그 사진들을 눈으로 봅니다 (멘토 피드백 5번)

**숫자로는 알 수 없는 걸 사진이 알려줍니다.**

30장에 공통점이 보이면 — 전부 털이 길다, 전부 실내 조명이다, 전부 피부가
접혀 있다 — 그건 **촬영 가이드나 데이터 정제로 잡히는 문제**입니다.
공통점이 없으면 모델을 더 손봐야 한다는 뜻이고요. 어느 쪽인지가 다음 실험을
정합니다.

In [ ]:
# 헛알림 — 정상인데 병원 보낸 사진. 확신이 센 것부터 봅니다
#   (모델이 헷갈린 게 아니라 **확신을 갖고 틀린** 것이 원인을 더 잘 보여줍니다)
fa = pv[(pv["label_orig"] == NORMAL_LABEL) & (pv["pred"] != NORMAL_LABEL)].copy()
fa = fa.sort_values("s1_score", ascending=False)
fa["note"] = fa.apply(lambda r: f"→{r['pred']} p={r['s1_score']:.2f}", axis=1)
print(f"헛알림 {len(fa):,}장 중 확신 상위 30장")

# ⚠️ 1단계가 본 크롭(f320)을 보여줍니다 — 2단계 크롭을 보면 딴 사진을 보는 셈입니다
errors.contact_sheet(fa.head(120), n=30, cols=6, seed=0,
                     title="헛알림 — 정상인데 '병원 가보세요' (1단계 입력 f320)",
                     save_to=W / "reports" / "false_alarms.png")

In [ ]:
# 놓친 병변 — 가장 위험한 오류. 이쪽은 확신이 센 순(= 정상이라고 강하게 믿은 순)
miss = pv[(pv["label_orig"] != NORMAL_LABEL) & (pv["pred"] == NORMAL_LABEL)].copy()
miss = miss.sort_values("s1_score", ascending=True)
miss["note"] = miss.apply(lambda r: f"실제 {r['label_orig']} p={r['s1_score']:.2f}", axis=1)
print(f"놓친 병변 {len(miss):,}장 중 가장 확신했던 30장")
errors.contact_sheet(miss.head(120), n=30, cols=6, seed=0,
                     title="놓친 병변 — 병변인데 '괜찮아요' (1단계 입력 f320)",
                     save_to=W / "reports" / "missed_lesions.png")

In [ ]:
# 1위 혼동 쌍 (병변끼리) 을 2단계 크롭으로 봅니다 — 형태를 봐야 하니까요
_lesion_pairs = [p for p in pairs_va
                 if p["true"] != NORMAL_LABEL and p["pred"] != NORMAL_LABEL]
if _lesion_pairs:
    top = _lesion_pairs[0]
    sel = pv[(pv["label_orig"] == top["true"]) & (pv["pred"] == top["pred"])].copy()
    sel = crop.switch_tag(sel, BEST_CROP, verbose=False)
    sel["note"] = f"{top['true']}→{top['pred']}"
    print(f"1위 병변 혼동: {top['true']} → {top['pred']}  {top['n']:,}장 "
          f"({top['share_of_true']:.1%})")
    errors.contact_sheet(sel, n=24, cols=6, seed=0,
                         title=f"{top['true']} 인데 {top['pred']} 라고 함 (m2.5)",
                         save_to=W / "reports" / "top_confusion.png")
else:
    print("병변끼리의 혼동 칸이 없습니다 (드문 경우입니다)")

## 6. Grad-CAM — 헛알림 난 사진에서 **어디를** 봤나

정상 사진을 병변이라고 했을 때 모델이 무엇을 보고 그랬는지입니다.
털뿌리·접힌 피부·조명 반사에 열이 몰려 있으면 원인이 특정됩니다.

In [ ]:
# 1단계 모델로 봅니다 — 헛알림을 만든 건 1단계니까요
_fa = fa.head(60).copy()
_fa["label"] = _fa["label_orig"]
explain.grid(m1, _fa, cfg=cfg, n=8, classes=CLASSES_STAGE1, seed=0)

## 7. 기록

⚠️ **여기서 나온 걸 보고 설정을 바꾸면, 그 설정의 성적은 val 로 다시 재야
합니다.** holdout 은 이미 STEP 10 에서 열었고, 이 노트북은 열어본 걸 **읽기만**
했습니다.

In [ ]:
out = W / "reports"
out.mkdir(parents=True, exist_ok=True)
payload = {
    "step": "STEP11_오답분석",
    "stage1": {"model": KEY1, "crop": STAGE1_CROP, "exp": EXP1},
    "stage2": {"model": KEY2, "crop": BEST_CROP, "exp": EXP2},
    "threshold": THR1, "img_size": IMG_SIZE,
    "val": {
        "final_macro_f1": rep_va["final_macro_f1"],
        "false_alarm_rate": rep_va["false_alarm_rate"],
        "screening_recall": rep_va["lesion_screening_recall"],
        "confusion": rep_va["confusion"],
        "confusion_pairs": pairs_va,
        "false_alarm_targets": fa_va,
        "miss_sources": ms_va,
    },
    "holdout_확인용": {
        "final_macro_f1": rep_ho["final_macro_f1"],
        "false_alarm_targets": fa_ho,
        "miss_sources": ms_ho,
    },
    "n_val": int(len(pv)), "n_holdout": int(len(ph)),
    "classes": stages.PIPELINE_CLASSES,
}
p = out / "step11_errors.json"
p.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
print("저장:", p)
print("그림:", *(f.name for f in sorted(out.glob("*.png"))))
print("\n다음 — 사진 30장에서 공통점이 보였나요?")
print("  보임  → 촬영 가이드 / 데이터 정제로 잡습니다 (재학습 전에)")
print("  안 보임 → 모델·입력을 손봐야 합니다 (04 백본 비교 결과와 같이 판단)")